# CSE2530 Computational Intelligence
## Assignment 1: Ant Colony Optimization and Genetic Algorithms

_Fill in your group number **from Brightspace**, names, and student numbers._

|  Group 35  |           Student number          |
|------------|----------------------|
| Belouaer, Zeyd  |        6160662       |,
| Dos Santos Roldão, Vasco  |        6127916       |
| Jacobs, Mats  |        6255698       |
| Kopmels, Kylian  |        6164846       |

#### Imports

In [5]:
"""
You may only use numpy to implement your algorithms
You can make use of any other libraries for miscellaneous functions, e.g. to create the visual aids.
Put all of your imports in this code block.
"""
import numpy as np
import random
import sys
import time

"""
The following classes are fully implemented in their own files and you should not change them.
Nonetheless, we encourage you to check how they work; this will help you get started.
"""
from Coordinate import Coordinate
from Direction import Direction
from PathSpecification import PathSpecification
from Route import Route
from SurroundingPheromone import SurroundingPheromone
from TSPData import TSPData

## Part 1: The Travelling Robot Problem
### 1.1 Problem Analysis
#### Question 1:

<div style="background-color:#f1be3e">

The traveling salesman problem (TSP) statement is as follows: 

We have a salesman traveling to sell items in different places. Given a graph G(V,E), every destination (including the salesman's home) is a vertex, and every path between 2 destinations is an edge. The solution is the collection of edges that represent the shortest path for the salesman to go to each city once and then back home.

Source: https://www.mathematics.pitt.edu/sites/default/files/TSP.pdf

#### Question 2

<div style="background-color:#f1be3e">

1. The output must include not only the smallest distance, but also the path we need to take.
2. There can be multiple ways between 2 products.
3. Distances are based on amount of steps in a maze, this is a pattern our genetic algorithm might use that isn't necessarily applicable to the classical TSP.

#### Question 3

<div style="background-color:#f1be3e">

The problem is NP-hard, meaning no efficient analytical problem exists. However, CI techniques exist that can gives us solutions close to the optimal solution. CI is useful for this task, as it is quite easy to create training data and corresponding labels for this problem, and it is also easy to define a loss/fitness function for this problem. Additionally, we can create a fitness function in such a way that similar solutions have a similar fitness so a genetic algorithm works here.
_Write your answer here._

### 1.2 Genetic Algorithm

In [ ]:
# TSP problem solver using genetic algorithms.
class GeneticAlgorithm:
    def calculate_fitness(self, generation,tsp_data):
        fitness = self.calculate_length(generation,tsp_data)
        return 1.0 / fitness
    
    def calculate_length(self, generation, tsp_data):
        mat = np.array(tsp_data.distances)
        fromitems = generation[:,:-1] #all elements but the last column (those items go to end, not to another item)
        toitems = generation[:,1:] #all elements but the first column (those items are preceded by start, not another item)
        fitnessthing = mat[fromitems,toitems] #create a matrix of the distances between from and to
        fitness = np.sum(fitnessthing, axis=1) #sum over all the rows (representing the chromosomes)
        fitness += (np.array(tsp_data.start_distances)[generation[:,0]] + np.array(tsp_data.end_distances)[generation[:,-1]]) #add the start and end distances for the first and last columns too
        return fitness
    
    def crossover(self, p1, p2):
        if np.random.rand() > 0.7:
            return p1.copy()

        size = len(p1)
        a, b = sorted(np.random.choice(size, 2, replace=False))

        child = [-1] * size
        child[a:b] = p1[a:b]

        ptr = 0
        for gene in p2:
         if gene not in child:
            while child[ptr] != -1:
                  ptr += 1
            child[ptr] = gene

        return np.array(child)
        


        
    def mutation(self, chromosome1): 
        #todo
        c1 = chromosome1.copy()
        if np.random.rand() > 0.1:
            return c1
        indices = np.random.choice(len(chromosome1),2,False)
        temp = c1[indices[0]]
        c1[indices[0]] = c1[indices[1]]
        c1[indices[1]] = temp
        return c1

    """
    Constructs a new 'genetic algorithm' object.
    @param generations: the amount of generations.
    @param pop_size: the population size.
    """
    def __init__(self, generations, pop_size):
        self.generations = generations
        self.pop_size = pop_size
    """
    This method should solve the TSP.
    @param tsp_data: the data describing the problem.
    @return the optimized product sequence.
    """
    def solve_tsp(self, tsp_data):
        random.seed(19)
        length = len(tsp_data.product_locations)
        generation = np.array([random.sample(range(length), length) for _ in range(self.pop_size)])
        children = np.zeros(generation.shape,dtype=int)
        for i in range(self.generations-1):
            fitness = self.calculate_fitness(generation, tsp_data)
            elite = generation[np.argmax(fitness)]
            children[0] = elite.copy()
            relative = fitness / np.sum(fitness)
            cumulative = np.cumsum(relative)
            for j in range(1,self.pop_size):
                index1 = np.searchsorted(cumulative, np.random.rand())
                index2 = np.searchsorted(cumulative, np.random.rand())
                child = self.mutation(self.crossover(generation[index1], generation[index2]))
                children[j] = child
            generation = children.copy()
        return generation[np.argmax(self.calculate_fitness(generation,tsp_data))]

#### Question 4

<div style="background-color:#f1be3e">

Each gene is an integer representing the ID of a specific item, each chromosome represents a list of item IDs in the order we take them (e.g. the chromosome [3,1,2] means "start at the start location, take product 3, then product 1, then product 2, and the IDs are genes) 

#### Question 5

<div style="background-color:#f1be3e">

We want to minimise the length, and our fitness function is maximized for "better" solutions (solutions with a smaller length), so it makes sense to take (path_length)^-1 as the fitness function.

#### Question 6

<div style="background-color:#f1be3e">

We will use roulette wheel selection (calculate the fitness ratio and pick randomly with a probability proportional to it)

#### Question 7

<div style="background-color:#f1be3e">

We will use crossover (take n objects from the first parent, the rest from the 2nd parent) and mutation (randomly swap 2 elements). Their function is to create variation in the children such that we converge to a better solution.

#### Question 8

<div style="background-color:#f1be3e">

Due to the nature of the TSP (swapping the order of 2 nodes can make the distance travelled much different) mutation can introduce chromosomes in very different regions of the fitness function, which can prevent local minima by introducing a solution with a much smaller length.

#### Question 9

<div style="background-color:#f1be3e">
Elitism refers to guaranteeing that (a subset of) top solutions stay in the pool, bypassing mutation chance and roulette wheel selection. We use this for the top 1 solution, because this guarantees that the next generation's top chromosome is at least as good as the current one, which is a guarantee we otherwise wouldn't have.

#### Question 10

In [53]:
# Please keep your parameters for the Genetic Algorithm easily changeable here
population_size = 200
generations = 200
persist_file = "./../data/optimal_tsp"

# Setup optimization
tsp_data = TSPData.read_from_file(persist_file)
ga = GeneticAlgorithm(generations, population_size)

# Run optimzation and write to file
solution = ga.solve_tsp(tsp_data)
print(ga.calculate_length(np.array([solution]), tsp_data))

tsp_data.write_action_file(solution, "./../data/tsp_solution.txt")

[1325]


<div style="background-color:#f1be3e">
Our total length is 1325. There is no way to tell the actual optimal answer, however the values for population_size and generations are absurdly high so t

## Part 2: Path Finding Through Ant Colony Optimization
### 2.2 Observing the Problem

#### Question 11

Ant Colony Optimization (ACO) is a technique used to solve optimization problems, like the traveling salesman problem. 

#### Question 12

Dead ends forces ants to backtrack, but they do leave pheromones in the dead end, so it causes a suboptimal solution.  
Loops in the maze are also probable causes for bad solutions. If an ant decides to run through a loop, then there will be pheromones in the loop. This will cause other ants to also check out the loop which reinforces a bad path.

#### Question 13

You want to punish long paths, so you divide by the total path length $L_i$.
$$\Delta\tau_i^k = \frac{Q}{L_i}$$
Variables:  
$\Delta\tau_i^k$: amount of pheromone dropped by ant $k$ on link $i$  
$L_i$: the length of the path that ant $k$ took to get back to the nest  
$Q$: a constant that represents how much pheromone is available  

Pheromones act as a kind of memory for the ants, without pheromones the ants would just keep going on random walks, which kind of defeats the purpose of using ACO.

#### Question 14

You want large amounts of pheromone to decay quickly in order to prevent good paths from becoming dominant too quickly. For example, if a path is by far the best one at the start, but the ants find a better one later on, then you want to switch to the better path. If pheromones evaporate very slowly, then old paths will dominate new paths which could prevent a better solution from emerging.  
You also want to evaporate low quantities of pheromones a lot slower, so the colony will always keep paths open. This prevents the colony from giving up on a path, which allows the ants to update the "bad" paths to make sure that it's still bad.

$$
\tau_{i_{t+1}} = (1-\rho) \cdot \tau_{i_t}
$$
Variables:  
$\tau_{i_{t+1}}$: the new amount of pheromones on link $i$ after evaporation  
$\rho$: the evaporation rate, $0$ means no evaporation, $1$ means instant evaporation  
$\tau_{i_t}$: the amount of pheromones in link $i$ before evaporation  

### 2.3 Implementing the Ant Algorithm

In [11]:
# Class that represents the basic Ant functionality
class StandardAnt:

    """
    Constructor of a StandardAnt taking a Maze and PathSpecification
    @param maze: the Maze where the ant will try to find a route
    @param path_specification: the PathSpecification consisting of a start and an end coordinate
    """
    def __init__(self, maze, path_specification):
        self.maze = maze
        self.start = path_specification.get_start()
        self.end = path_specification.get_end()
        self.current_position = self.start
        self.rand = random

    """
    Method that performs a single complete run through the maze by the ant
    @return the route found by the ant
    """
    def find_route(self):
        route = Route(self.start)
        pass


In [12]:
# Class that holds all of the maze data.
# This includes the pheromones, the open and blocked tiles in the system,
# and the starting and end coordinates for the ants.
class Maze:

    """
    Constructor of a Maze
    @param walls: array of ints representing the accessible (1) and inaccessible (0) tiles
    @param width: the width (horizontal dimension) of the Maze
    @param length: the length (vertical dimension) of the Maze
    """
    def __init__(self, walls, width, length):
        self.walls = walls
        self.length = length
        self.width = width
        self.start = None
        self.end = None
        self.initialize_pheromones()

    """
    Initialize pheromones on all tiles of the Maze
    """
    def initialize_pheromones(self):
        pass

    """
    Reset the Maze for a new shortest path problem
    """
    def reset(self):
        self.initialize_pheromones()

    """
    Update the pheromones along a certain route according to a certain Q
    @param route: the route taken by an ant
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_route(self, route, q):
        pass

    """
    Update pheromones for a list of routes
    @param routes: a list of routes taken by the ants
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_routes(self, routes, q):
        for r in routes:
            self.add_pheromone_route(r, q)

    """
    Evaporate pheromone
    @param rho: the evaporation factor
    """
    def evaporate(self, rho):
        pass

    """
    Getter for the width of the maze
    @return the width of the maze
    """
    def get_width(self):
        return self.width

    """
    Getter for the length of the maze
    @return the length of the maze
    """
    def get_length(self):
        return self.length

    """
    Returns a the amount of pheromones on the neighbouring positions (N/S/E/W)
    @param position: the coordinate where we need to check the surrounding pheromones
    @return the pheromones on the neighbouring coordinates.
    """
    def get_surrounding_pheromone(self, position):
        pass

    """
    Getter for the pheromones on a specific coordinate.
    If the position is not in bounds returns 0
    @param pos: coordinate for the poition of interest
    @return the amount of pheromone at the specified poition
    """
    def get_pheromone(self, pos):
        pass

    """
    Check whether a coordinate lies in the bounds of the current maze
    @param position: the position that we need to check
    @return true if the coordinate lies within the current maze
    """
    def in_bounds(self, position):
        return position.x_between(0, self.width) and position.y_between(0, self.length)

    """
    Representation of Maze as defined by the input file format.
    @return the human-readable representation of a maze
    """
    def __str__(self):
        string = ""
        string += str(self.width)
        string += " "
        string += str(self.length)
        string += " \n"
        for y in range(self.length):
            for x in range(self.width):
                string += str(self.walls[x][y])
                string += " "
            string += "\n"
        return string

    """
    Method that builds a maze from a file
    @param file_path: path to the file which stores the maze
    @return a maze object with pheromones initialized to 0s on inaccessible and 1s on accessible tiles
    """
    @staticmethod
    def create_maze(file_path):
        try:
            f = open(file_path, "r")
            lines = f.read().splitlines()
            dimensions = lines[0].split(" ")
            width = int(dimensions[0])
            length = int(dimensions[1])
            
            #make the maze_layout
            maze_layout = []
            for x in range(width):
                maze_layout.append([])
            
            for y in range(length):
                line = lines[y+1].split(" ")
                for x in range(width):
                    if line[x] != "":
                        state = int(line[x])
                        maze_layout[x].append(state)
            print("Ready reading maze file " + file_path)
            return Maze(maze_layout, width, length)
        except FileNotFoundError:
            print("Error reading maze file " + file_path)

In [13]:
# Class representing the complete ACO algorithm.
# Finds shortest path between two points in a maze according to a path specification.
class AntColonyOptimization:

    """
    Constructs a new optimization object using the ant algorithm
    @param maze: the maze (environment) for ants
    @param ants_per_gen: the number of ants per generation (between update of pheromones)
    @param generations: the total number of generations of ants (pheromone updates)
    @param q: the normalization factor for the amount of dropped pheromone
    @param evaporation: the evaporation factor for the pheromones
    """
    def __init__(self, maze, ants_per_gen, generations, q, evaporation):
        self.maze = maze
        self.ants_per_gen = ants_per_gen
        self.generations = generations
        self.q = q
        self.evaporation = evaporation

    """
    Loop that starts the shortest path process
    @param path_specification: description of the route we wish to optimize
    @return the optimized route according to the ACO algorithm
    """
    def find_shortest_route(self, path_specification):
        self.maze.reset()
        pass

In [14]:
# Please keep your parameters for the ACO easily changeable here
gen = 1
no_gen = 1
q = 1600
evap = 0.1

# Construct the optimization objects
maze = Maze.create_maze("./../data/hard_maze.txt")
spec = PathSpecification.read_coordinates("./../data/hard_coordinates.txt")
aco = AntColonyOptimization(maze, gen, no_gen, q, evap)

start_time = int(round(time.time() * 1000))
shortest_route = aco.find_shortest_route(spec)

print("Time taken: " + str((int(round(time.time() * 1000)) - start_time) / 1000.0))
print("Route size: " + str(shortest_route.size()))

shortest_route.write_to_file("./../data/hard_solution.txt")

Ready reading maze file ./../data/hard_maze.txt
Time taken: 0.0


AttributeError: 'NoneType' object has no attribute 'size'

### 2.4 Upgrading Your Ants with Intelligence

#### Question 15

In [15]:
# Class that represents the intelligent Ant
class IntelligentAnt:

    """
    Constructor of an IntelligentAnt taking a Maze and PathSpecification
    @param maze: the Maze where the ant will try to find a route
    @param path_specification: the PathSpecification consisting of a start and an end coordinate
    """
    def __init__(self, maze, path_specification):
        self.maze = maze
        self.start = path_specification.get_start()
        self.end = path_specification.get_end()
        self.current_position = self.start
        self.rand = random

    """
    Method that performs a single complete run through the maze by the ant
    @return the route found by the ant
    """
    def find_route(self):
        route = Route(self.start)
        pass


<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

### 2.5 Parameter Optimization

#### Question 16

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 17

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.6 The Final Route

#### Question 18

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.7 Synthesis

#### Question 19

In [0]:
# Please keep your parameters for the synthesis part easily changeable here
gen = 1
no_gen = 1
q = 1000
evap = 0.1

persist_file = "./../tmp/my_tsp"
tsp_path = "./../data/tsp_products.txt"
coordinates = "./../data/hard_coordinates.txt"

# Construct optimization
maze = Maze.create_maze("./../data/hard_maze.txt")
tsp_data = TSPData.read_specification(coordinates, tsp_path)
aco = AntColonyOptimization(maze, gen, no_gen, q, evap)

# Run optimization and write to file
tsp_data.calculate_routes(aco)
tsp_data.write_to_file(persist_file)

# Read from file and print
tsp_data2 = TSPData.read_from_file(persist_file)
print(tsp_data == tsp_data2)

# Solve TSP using your own paths file
ga = GeneticAlgorithm(generations, population_size)
solution = ga.solve_tsp(tsp_data2)
tsp_data2.write_action_file(solution, "./../data/tsp_solution.txt")

<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

## Part 3: Open Questions
### 3.1 Reflection

#### Question 20

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 21

<div style="background-color:#f1be3e">

_Write your answer here._

### 3.2 Pen and Paper

#### Question 22

<div style="background-color:#f1be3e">

_Write your answer here. You can also choose to simply include a photo of your solution._

### 3.3 Division of Work

#### Question 23

<div style="background-color:#f1be3e">


|          Component          |  Name A   |  Name B   |  Name C   |  Name D   |
|-----------------------------|-----------|-----------|-----------|-----------|
| Code (design)               |     A     |     B     |     C     |     D     |
| Code (implementation)       |     A     |     B     |     C     |     D     |
| Code (validation)           |     A     |     B     |     C     |     D     |
| Experiments (execution)     |     A     |     B     |     C     |     D     |
| Experiments (analysis)      |     A     |     B     |     C     |     D     |
| Experiments (visualization) |     A     |     B     |     C     |     D     |
| Report (original draft)     |     A     |     B     |     C     |     D     |
| Report (reviewing, editing) |     A     |     B     |     C     |     D     |

### References

<div style="background-color:#f1be3e">

**If you made use of any non-course resources, cite them below.**